In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

train_df = pd.read_csv('dataset/splits_sample/train.csv')
test_df = pd.read_csv('dataset/splits_sample/test.csv')

train_df['content_clean_stem'] = train_df['content_clean_stem'].fillna('')
test_df['content_clean_stem'] = test_df['content_clean_stem'].fillna('')

label_mapping = {
    'reliable': 0, 'political': 0, 'bias': 0,
    'fake': 1, 'conspiracy': 1, 'rumor': 1, 'clickbait': 1, 
    'junksci': 1, 'unreliable': 1, 'hate': 1
}

train_df = train_df[~train_df['type'].isin(['unknown', 'satire'])].copy()
test_df = test_df[~test_df['type'].isin(['unknown', 'satire'])].copy()

train_df['label'] = train_df['type'].map(label_mapping)
test_df['label'] = test_df['type'].map(label_mapping)

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df['content_clean_stem'])
X_test_tfidf = tfidf.transform(test_df['content_clean_stem'])

y_train = train_df['label']
y_test = test_df['label']

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.92      0.89     31754
           1       0.92      0.85      0.88     30624

    accuracy                           0.89     62378
   macro avg       0.89      0.89      0.89     62378
weighted avg       0.89      0.89      0.89     62378



In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

print("--- Naive Bayes Evaluation ---")
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_y_pred = nb_model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, nb_y_pred):.4f}\n")
print(classification_report(y_test, nb_y_pred))

--- Naive Bayes Evaluation ---
Accuracy: 0.8302

              precision    recall  f1-score   support

           0       0.83      0.84      0.83     31754
           1       0.83      0.82      0.83     30624

    accuracy                           0.83     62378
   macro avg       0.83      0.83      0.83     62378
weighted avg       0.83      0.83      0.83     62378



In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

print("--- LIAR Dataset Diagnostic & Evaluation ---")

liar_cols = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job', 'state', 'party', 
    'barely_true', 'false_count', 'half_true', 'mostly_true', 'pants_on_fire', 'context'
]
liar_test = pd.read_csv('test.tsv', sep='\t', names=liar_cols)

print(f"Initial rows: {len(liar_test)}")

liar_test['label'] = liar_test['label'].astype(str).str.strip().str.lower()

liar_mapping = {
    'pants-fire': 1, 'false': 1, 'barely-true': 1,
    'half-true': 0, 'mostly-true': 0, 'true': 0
}

liar_test['binary_label'] = liar_test['label'].map(liar_mapping)

mapped_count = liar_test['binary_label'].notna().sum()
print(f"Successfully mapped labels: {mapped_count}")

liar_test = liar_test.dropna(subset=['statement', 'binary_label'])
print(f"Valid rows for testing: {len(liar_test)}\n")

if len(liar_test) == 0:
    print("Error: Dataset is empty after cleaning.")
else:
    print("Extracting TF-IDF features...")
    X_liar_tfidf = tfidf.transform(liar_test['statement'])
    y_liar_true = liar_test['binary_label']

    y_liar_pred = model.predict(X_liar_tfidf)

    print("--- Cross-Domain Evaluation Results ---")
    print(f"Accuracy: {accuracy_score(y_liar_true, y_liar_pred):.4f}\n")
    print(classification_report(y_liar_true, y_liar_pred))

--- LIAR Dataset Diagnostic & Evaluation ---
Initial rows: 1267
Successfully mapped labels: 1267
Valid rows for testing: 1267

Extracting TF-IDF features...
--- Cross-Domain Evaluation Results ---
Accuracy: 0.5493

              precision    recall  f1-score   support

           0       0.57      0.85      0.68       714
           1       0.45      0.16      0.24       553

    accuracy                           0.55      1267
   macro avg       0.51      0.51      0.46      1267
weighted avg       0.52      0.55      0.49      1267

